In [ ]:
import os
import sys
import json
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

import numpy as np  # Ensure this is imported if not already


import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = torch.Generator().manual_seed(42)
print(device)

cuda


In [5]:
from data.cifar10 import get_cifar10_pipeline

train_loader, val_loader, test_loader = get_cifar10_pipeline(batch_size=128, indexed=True)
sample_x, sample_y, idx = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)
print(idx.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([128, 3, 32, 32])
torch.Size([128])
torch.Size([128])


In [ ]:
from models.resnet import resnet34, resnet50
from utils.pipeline import evaluate_model
from utils.summary import get_model_stats


model = resnet34().to(device)    # try to get this to work for 5-10 epochs then I can swap it out and run it later

63

In [ ]:
from utils.pipeline import evaluate_model
from utils.losses import DistillationLoss

# define in self_distill_model with arg for T
distill_criterion = DistillationLoss(T=2.0, alpha=0.7)
# define external to train_val as arg passed 
normal_criterion = nn.CrossEntropyLoss()

def self_distill_model(train_loader, model, criterion, optimizer, scheduler, device, teacher_logits_dict):
    model.train()
    epoch_loss = []

    for inputs, labels, indices in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        # Determine if we can apply self-distillation
        if all(idx.item() in teacher_logits_dict for idx in indices):
            prev_logits = torch.stack([teacher_logits_dict[idx.item()] for idx in indices]).to(device)
            loss = criterion(outputs, prev_logits.detach(), labels) # swap with distill_criterion student (current), teacher (past), target
        else:
            loss = F.cross_entropy(outputs, labels)  # swap with normal_criterion

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store current logits for self-distillation in next epoch
        for i, idx in enumerate(indices):
            teacher_logits_dict[idx.item()] = outputs[i].detach().clone()

        epoch_loss.append(loss.item())

    scheduler.step()
    return epoch_loss


def train_val(train_loader, val_loader, model, criterion, optimizer, scheduler, device, aux_metrics, path,
              patience=10, epochs=200):

    metrics = {"train_loss": [], "val_loss": []}
    for k in aux_metrics.keys():
        metrics[k] = []

    best_val_acc = 0
    counter = 0
    teacher_logits_dict = {}

    for epoch in range(epochs):
        train_loss = self_distill_model(train_loader, model, criterion, optimizer, scheduler, device, teacher_logits_dict)
        val_loss = evaluate_model(val_loader, model, criterion, device)

        metrics['train_loss'].append(np.mean(train_loss))
        metrics['val_loss'].append(np.mean(val_loss))

        for k, v in aux_metrics.items():
            stat = evaluate_model(val_loader, model, v, device)
            metrics[k].append(np.mean(stat))

        # Replace 'accuracy' with whatever your primary metric is
        if metrics['accuracy'][-1] >= best_val_acc:
            best_val_acc = metrics['accuracy'][-1]
            counter = 0
            print(f"Epoch {epoch+1}: New best accuracy: {metrics['accuracy'][-1]:.4f}, saving model...")
            state = {
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'val_loss': metrics['val_loss'][-1],
                'accuracy': metrics['accuracy'][-1]
            }
            torch.save(state, path)
        else:
            counter += 1

        if counter >= patience:
            print(f"Epoch {epoch+1}: Early stop triggered.")
            break

    return metrics


In [ ]:
def self_distill():
    distill_criterion = DistillationLoss(T=2.0, alpha=0.7)
    teacher_logits_dict = {}  # Maps sample indices to logits

    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (inputs, labels, indices) in enumerate(dataloader):  # `indices` are sample ids
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs)

            # Get teacher logits from previous epoch, if available
            if all(idx.item() in teacher_logits_dict for idx in indices):
                # Stack teacher logits from stored values
                prev_logits = torch.stack([teacher_logits_dict[idx.item()] for idx in indices]).to(device)
                loss = distill_criterion(outputs, prev_logits.detach(), labels)
            else:
                # Fall back to standard cross-entropy for new samples
                loss = F.cross_entropy(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Store current logits for next epoch
            for i, idx in enumerate(indices):
                teacher_logits_dict[idx.item()] = outputs[i].detach().clone()